<a href="https://colab.research.google.com/github/shivanshi-09/Introduction-to-Machine-Learning-/blob/main/A1_B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes

dataset = load_diabetes(as_frame=True).frame
x = np.array(dataset.drop(columns=['target']))
#442 samples and 10 features
y = np.array(dataset['target'])

rng = np.random.default_rng(42)
idx = rng.permutation(len(x))
split = int(0.8* len(y))
x_train, x_test = x[idx[:split]], x[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

### Decision Tree Regression

A decision tree regression model partitions the feature space into a set of regions by recursively splitting the data based on feature values. At each internal node, the algorithm selects a feature $j$ and a threshold $t$ to split the data into two subsets:

$$
x_j \le t \quad \text{and} \quad x_j > t
$$

The goal of each split is to reduce the prediction error within the resulting subsets.

For regression trees, the prediction at each leaf node is the average value of the target variable for the samples that fall into that region:

$$
\hat{y} = \frac{1}{|R|} \sum_{i \in R} y_i
$$

where $R$ denotes the set of samples assigned to the leaf node.

In [ ]:
class Node:
  def __init__(self, feature = None, threshold = None, left = None, right = None, *, value = None):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value

class DecisionTree:
  def __init__(self, max_depth= None, min_samples_split= 2):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.root = None

  def _mse(self, y):
    if len(y)==0:
      return 0
    return np.mean((y-np.mean(y))**2)

  def best_split(self, x, y):
    best_feature = None
    best_threshold = None
    best_loss = float("inf")

    n_samples, n_features = x.shape

    #go through all 10 features
    for feature in range(n_features):
      sorted_indices = np.argsort(x[:, feature]) #returns indices that would sort the array (ascending)

      x_feature_sorted = x[sorted_indices, feature] #only values of current feature sorted
      y_sorted = y[sorted_indices] #target values sorted

      left_count = 0
      right_count = n_samples

      left_sum = 0.0
      right_sum = np.sum(y_sorted)

      left_sq_sum = 0.0
      right_sq_sum = np.sum(y_sorted**2)

      for i in range (1, n_samples):
        #simulating all possible binary split ploints
        yi = y_sorted[i -1]
        left_count += 1
        right_count -= 1
        left_sum += yi
        right_sum -= yi
        left_sq_sum += yi**2
        right_sq_sum -= yi**2

        if x_feature_sorted[i] == x_feature_sorted[i-1]:
          continue

        if left_count == 0 or right_count == 0:
            continue

        #getting the variance
        left_mse = (left_sq_sum/left_count) - (left_sum/left_count)**2
        right_mse = (right_sq_sum/right_count) - (right_sum/right_count)**2

        loss = left_count * left_mse + right_count * right_mse

        if loss < best_loss:
          best_loss = loss
          best_feature = feature
          best_threshold = (x_feature_sorted[i] + x_feature_sorted[i-1]) / 2

    return best_feature, best_threshold

  def _build(self, x, y, depth):
    if len(y) < self.min_samples_split:
      return Node(value = np.mean(y))
    if self.max_depth is not None and depth >= self.max_depth:
      return Node(value = np.mean(y))

    feature, threshold = self.best_split(x, y)

    if feature is None:
      return Node(value = np.mean(y))

    left_mask = x[:, feature] <= threshold
    right_mask = x[:, feature] > threshold

    left = self._build(x[left_mask], y[left_mask], depth + 1)
    right = self._build(x[right_mask], y[right_mask], depth + 1)

    return Node(feature, threshold, left, right)

  def build_tree(self, x, y):
    self.root = self._build(x, y, 0)

  def _predict_one(self, x_sample, node):
    if node.value is not None:
      return node.value
    if x_sample[node.feature] <= node.threshold:
      return self._predict_one(x_sample, node.left)
    else:
      return self._predict_one(x_sample, node.right)

  def predict(self, X_test):
    return np.array([self._predict_one(i, self.root) for i in X_test])

### Mean Squared Error Split Criterion

To determine the best split, the algorithm minimizes the Mean Squared Error (MSE) within the resulting subsets. The MSE for a set of targets $y$ is defined as

$$
\text{MSE}(y) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2
$$

where $\bar{y}$ is the mean of the target values.

For a candidate split producing left and right subsets, the total loss is computed as the weighted sum of the errors:

$$
L = n_L \cdot \text{MSE}(y_L) + n_R \cdot \text{MSE}(y_R)
$$

where:
- $n_L$ and $n_R$ are the number of samples in the left and right nodes
- $y_L$ and $y_R$ are the target values in each subset.

The split that minimizes this loss is selected.

In [ ]:
best_split = float('inf')

print(f"{'Max Depth':<12} {'Min Split':<12} {'Train MSE':>10} {'Test MSE':>10}")
print("-" * 48)

for depth in [3, 5, 7, 10, None]:
    for min_split in [2, 5, 10, 20, 30]:
        tree = DecisionTree(max_depth=depth, min_samples_split=min_split)
        tree.build_tree(x_train, y_train)
        train_mse = np.mean((tree.predict(x_train) - y_train) ** 2)
        test_mse  = np.mean((tree.predict(x_test)  - y_test)  ** 2)
        print(f"{str(depth):<12} {min_split:<12} {train_mse:>10.2f} {test_mse:>10.2f}")

    if test_mse < best_split:
      best_split = test_mse
      best_depth = depth
      best_min_split = min_split
      best_mse = test_mse

print(f"\n Best Depth: {best_depth}   Best Split:{best_min_split}   Best MSE: {best_mse}")

Max Depth    Min Split     Train MSE   Test MSE
------------------------------------------------
3            2               2965.44    3453.24
3            5               2965.44    3453.24
3            10              2965.44    3453.24
3            20              2965.44    3453.24
3            30              2965.44    3453.24
5            2               1938.71    4817.07
5            5               1993.93    4595.25
5            10              2040.44    4585.62
5            20              2087.89    4521.99
5            30              2277.86    4493.91
7            2                928.79    4926.07
7            5               1075.90    4587.83
7            10              1263.16    4421.55
7            20              1653.00    4510.60
7            30              2128.52    4288.30
10           2                305.36    5292.43
10           5                551.71    4977.34
10           10               882.09    4683.86
10           20              1510.20   

### Random Forest Regression

Random Forest is an ensemble learning method that combines multiple decision trees to improve predictive performance and reduce overfitting.

Each tree in the forest is trained on a bootstrap sample of the training data (sampling with replacement). In addition, at each split the algorithm considers only a random subset of the available features.

Given predictions from $T$ individual trees:

$$
\hat{y}_1(x), \hat{y}_2(x), \dots, \hat{y}_T(x)
$$

the final prediction of the random forest is obtained by averaging the predictions:

$$
\hat{y}(x) = \frac{1}{T} \sum_{t=1}^{T} \hat{y}_t(x)
$$

This averaging reduces variance and generally produces more stable predictions compared to a single decision tree.

In [ ]:
class RandomForest:
  def __init__ (self, n_trees = 100, max_depth = 3, min_samples_split = 5,
                n_features = None, sample_ratio = 0.8, random_seed = 42):
    self.n_trees = n_trees
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.n_features = n_features
    self.sample_ratio = sample_ratio
    self.random_seed = random_seed
    self.trees = []
    self.tree_features = []

  def build_forest (self, x, y):
    n_samples, n_total_features = x.shape
    #number of features per split
    n_feat = self.n_features or max(1, int(np.floor(np.sqrt(n_total_features))))
    #samples to bootstrap for each tree
    n_sub = max(1, int(self.sample_ratio * n_samples))

    rng = np.random.default_rng(self.random_seed)
    self.trees = []
    self.tree_features = []

    for i in range(self.n_trees):
      #bagging, randomly selects n_sub sample indices w replacement
      row_idx = rng.choice(n_samples, size= n_sub, replace = True)
      #selections n_feat features w out replacement
      feature_idx = rng.choice(n_total_features, size= n_feat, replace = False)
      #create a subset of the data for the tree
      x_sub = x[row_idx][:, feature_idx]
      y_sub = y[row_idx]
      tree = DecisionTree(max_depth=self.max_depth, min_samples_split= self.min_samples_split)
      tree.build_tree(x_sub, y_sub)
      self.trees.append(tree)
      self.tree_features.append(feature_idx)

  #uses the ensemble of trained trees to generate predictions for new data x
  def predict(self, x):
    all_preds = np.array ([
        tree.predict(x[:, feature_idx])
        for tree, feature_idx in zip(self.trees, self.tree_features)
    ])
    return np.mean(all_preds, axis = 0)

In [ ]:
best_rf_mse = float('inf')

print(f"{'n feat':<8} {'n trees':<10} {'Test MSE':>10}")
print("-" * 30)

for n_feat in [4, 5, 6, 7]:
    for n_trees in [50, 100, 200]:
        rf = RandomForest(n_trees=n_trees, max_depth=3, min_samples_split=2,
                          n_features=n_feat, sample_ratio=0.8, random_seed=42)
        rf.build_forest(x_train, y_train)
        test_mse = np.mean((rf.predict(x_test) - y_test) ** 2)
        print(f"{n_feat:<8} {n_trees:<10} {test_mse:>10.2f}")
        if test_mse < best_rf_mse:
            best_rf_mse = test_mse
            best_n_trees = n_trees
            best_n_feat = n_feat

print(f"\n Single Tree - Test MSE: {best_mse:7.2f}")
print(f"Random Forest - Test MSE: {best_rf_mse:.2f}")

n feat   n trees      Test MSE
------------------------------
4        50            3289.47
4        100           3323.74
4        200           3390.42
5        50            3253.21
5        100           3218.71
5        200           3234.95
6        50            3051.84
6        100           3027.52
6        200           3064.34
7        50            3146.00
7        100           3001.93
7        200           2975.59

 Single Tree - Test MSE: 3453.24
Random Forest - Test MSE: 2975.59
